# Module 3.1: Build a Grounded Booking Agent with GraphRAG

**Overview**

This notebook turns two GraphRAG read paths from Module 2 into a local booking agent. The agent reads hotel facts from Neo4j. A separate booking command checks rules and saves requests.

- **GraphRAG:** Answer a question with facts retrieved from a graph and its source text.
- **Passage tool:** Find source text and connected facts about a named hotel or policy.
- **Record tool:** Count, filter, rank, or average saved hotel records.
- **Grounding result:** State whether Neo4j contains enough information for an answer.
- **Guest limit:** Reject a booking request for more than 10 guests.
- **Safe retry:** Return the first result when the same booking request arrives again.

Use your configured Neo4j database and Amazon Bedrock. This notebook creates no AWS resources.

## Divide the work between Neo4j and AWS

| Neo4j | AWS |
|---|---|
| Stores hotel facts, source text, and booking requests | Amazon Bedrock chooses a tool and writes the final answer |
| Searches hotel text with vector and full-text indexes | Amazon Nova 2 creates the search vector for a question |
| Connects each matching source passage to its hotel | An Amazon Bedrock model writes Cypher for record questions |
| Enforces the guest limit and unique request IDs |  |

## Load the shared workshop code

Run the next two cells to load the workshop helpers and check your connections.

- **Neo4j ready:** Run cells that read or write graph data.
- **Bedrock ready:** Run cells that call an Amazon Bedrock model.
- **Retrieval ready:** Run the agent examples when both Neo4j and Bedrock are ready.

A live example prints a skip message when its required connection is unavailable.

In [ ]:
import os
import sys
from pathlib import Path

# The shared workshop/ package lives in notebooks/. These lines find that
# directory and put it on the import path. Everything else this notebook
# needs to start is in workshop/bootstrap.py.
_here = Path.cwd().resolve()
_named = os.environ.get("WORKSHOP_NOTEBOOKS_DIR") or _here
_starts = (Path(_named).expanduser().resolve(), _here, _here / "notebooks")
for _candidate in (*_starts, *_here.parents):
    if (_candidate / "workshop" / "bootstrap.py").is_file():
        sys.path.insert(0, str(_candidate))
        break

from workshop.bootstrap import start_module

NOTEBOOKS_ROOT, REPO_ROOT, MODULE_DIR = start_module("03-grounded-booking-agent")
print(f"Workshop root: {REPO_ROOT}")


In [ ]:
import inspect
import json
import os
import uuid
from datetime import date, timedelta

import boto3

from workshop.agent_tools import PASSAGE_TOOL, READ_TOOLS, RECORD_TOOL
from workshop.aws_region import configure_aws_region
from workshop.bedrock_providers import default_model_id
from workshop.contracts import (
    MAX_GUESTS,
    OVER_LIMIT_GUESTS,
    ReservationReason,
    ReservationStatus,
)
from workshop.fixtures import (
    HERO_NAME,
    HERO_SOURCE,
    apply_reservation_fixtures,
    load_manifest,
    readiness_problems,
)
from workshop.grounding import MISSING_LIVE_AVAILABILITY
from workshop.hybrid_retrieval import Neo4jConfig, search_hotel_knowledge
from workshop.prompts import BASE_GROUNDING_PROMPT
from workshop.retrieval_setup import report_problems
from workshop.workshop_utils import (
    ToolTraceHook,
    selected_tool_names,
    show_result,
)
from reservation_command import create_reservation_request
from neo4j import GraphDatabase

NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_READY = all(os.getenv(name) for name in NEO4J_ENV)
BEDROCK_READY = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = NEO4J_READY and BEDROCK_READY

AWS_REGION = configure_aws_region()
MODEL_ID = default_model_id()
HERO_QUESTION = f"What amenities and guest rating does {HERO_NAME} have?"
AVAILABILITY_QUESTION = f"Does {HERO_NAME} guarantee room availability next weekend?"

if not NEO4J_READY:
    print("Neo4j is not configured, so live cells will be skipped. Set NEO4J_URI/USERNAME/PASSWORD/DATABASE.")
if not BEDROCK_READY:
    print("AWS credentials are not configured, so live cells will be skipped.")
if RETRIEVAL_READY:
    print("Participant retrieval is configured. Ready to retrieve the fixture hotel.")

## 1. Prepare the graph for hotel search and booking

**Purpose:** Add the setup data and rules used by the booking examples.

Run the next cell before the search and booking examples. You can safely run it more than once.

- **Hotel ID:** A stable internal ID that connects a booking request to the correct hotel.
- **Uniqueness constraints:** Database rules that require unique IDs for hotels, booking requests, and the guest-limit rule.
- **Guest-limit rule:** A database rule that allows up to 10 guests in one request.
- **Readiness check:** A check that the example hotel exists and both search indexes are ready.

The setup leaves all source facts unchanged, including hotel names, amenities, and ratings.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph preparation: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        driver.verify_connectivity()
        problems = apply_reservation_fixtures(driver, config.database, manifest)
        if not problems:
            problems = readiness_problems(driver, config.database, manifest)
    finally:
        driver.close()
    if problems:
        print("Graph is not ready:")
        for problem in problems:
            print(f"  - {problem}")
    else:
        print("Participant graph is ready: both indexes online, fixtures applied, rule present.")

## 2. Search for one hotel's amenities and rating

> **What amenities and guest rating does AnyCompany Cairo Nile View have?**

**Purpose:** Find one hotel's facts and the source text that supports them.

Run the next cell to search in two ways. Exact-word search matches the hotel name. Meaning-based search finds text about amenities and ratings. Neo4j follows the matching text to the hotel and returns its facts.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping hotel details question: retrieval is not configured.")
else:
    results = search_hotel_knowledge(HERO_QUESTION)
    top = results[0]
    print(f"Question: {HERO_QUESTION}\n")
    print(f"Hotel: {top['hotel_name']} | hotel_id={top['hotel_id']}")
    print(f"Guest rating: {top['guest_rating']}")
    print(f"Amenities: {', '.join(top['amenities'])}")
    print("\nSource text that supports these facts:")
    print(top["chunk_text"][:600])

### Read the search result

The result combines source text with saved graph facts.

- **Source passage:** The original hotel text that supports the result.
- **Graph facts:** The hotel's amenities and guest rating.
- **`hotel_id`:** The stable internal ID used by booking requests.

The search uses the same settings on every run, so each result has the same fields. The next section gives the agent two tools that return this structured evidence.

## 3. Define the agent's two graph tools

**Purpose:** Give the agent one tool for source text and one tool for graph records.

- **`search_hotel_passages`:** Find source text and connected facts for a named hotel or policy.
- **`query_hotel_records`:** Count, filter, rank, or average saved hotel records.

### Follow one question through the agent

1. You ask a question.
2. The model reads each tool's name, description, and input schema.
3. The model chooses a tool and sends it the question.
4. Strands runs the tool and returns the result to the model.
5. The model writes an answer using the result.

A hotel question requires a tool. A social message can receive a direct reply.

### Read the Strands terms

**Brief overview**

- **`Agent`:** Send a question to the model and run the tool it selects.
- **`BedrockModel`:** Connect the agent to an Amazon Bedrock model.
- **`@tool`:** Mark a Python function as a tool the model can select.
- **`ToolTraceHook`:** Record each tool call and its result.

The next cell prints both tool specifications and the Python wrapper for the first tool.

In [ ]:
print(f"The model receives {len(READ_TOOLS)} tool specifications.\n")
for read_tool in READ_TOOLS:
    print(json.dumps(read_tool.tool_spec, indent=2))
    print()

print("The wrapper behind the first specification:\n")
print(inspect.getsource(READ_TOOLS[0]))

### Read the tool input and result

The model uses the printed name, description, and input schema to select a tool. Both tools reject a blank question.

Both tools return these main fields:

- **`ok`:** State whether the tool completed its work.
- **Evidence:** Return either source passages or graph records.
- **`grounding_result`:** State whether the evidence can answer the question.
- **`missing_fact`:** Name missing information inside `grounding_result`.

### Follow a record question

`query_hotel_records` needs an extra model call to write Cypher.

1. The agent model selects `query_hotel_records`.
2. The tool asks a model to write one read-only Cypher query.
3. Neo4j checks the query plan and runs a query that only reads data.
4. The tool returns the Cypher query and its records.
5. The agent model uses those records to answer the question.

Read the returned Cypher before accepting the answer. A valid query can still represent the question incorrectly.

### Handle empty results and errors

- **Empty `records`:** The query ran and found no matching records. Check the Cypher before deciding that the graph lacks the answer.
- **Error result:** The tool could not create or run a safe read query. It returns `ok: false` with an error code.

The next cell builds a new agent and trace for each question. It also prints the system prompt used by every agent.

In [ ]:
from strands import Agent
from strands.models import BedrockModel


def build_agent():
    """Build a new agent and trace for one question.

    Each routing example uses a new agent. This removes messages from earlier
    questions. Each trace then contains only the current question's calls.
    """
    trace = ToolTraceHook()
    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION),
        tools=list(READ_TOOLS),
        system_prompt=BASE_GROUNDING_PROMPT,
        hooks=[trace],
    )
    return agent, trace


def recorded_payloads(trace):
    """Return the complete JSON payload of every tool call a trace recorded."""
    return [
        payload
        for call in trace.calls
        for payload in call["payloads"]
        if isinstance(payload, dict)
    ]


print("System prompt sent with every question:\n")
print(BASE_GROUNDING_PROMPT)

## 4. Test the agent's tool choice

**Purpose:** Check whether each question reaches the expected tool.

Run the next cell. Each question starts with a new agent, so earlier messages cannot influence its choice. The output shows the selected tool and its result.

| Question | Expected tool |
|---|---|
| Amenities and guest rating for one named hotel | `search_hotel_passages` |
| Average guest rating of the hotels in Paris | `query_hotel_records` |
| Count of hotels that offer a spa | `query_hotel_records` |
| The recorded wording of a cancellation policy | `search_hotel_passages` |

The tool descriptions include the Paris average and spa count as examples. Add a different count, filter, or average question to test new wording.

Tool choice can vary between runs. A mismatch shows that the tool name or description needs clearer guidance.

In [ ]:
ROUTING_CASES = (
    (HERO_QUESTION, PASSAGE_TOOL),
    ("What is the average guest rating of the hotels in Paris?", RECORD_TOOL),
    ("How many hotels offer a spa?", RECORD_TOOL),
    (
        f"What is the cancellation policy at {HERO_NAME}? "
        "Quote the recorded wording.",
        PASSAGE_TOOL,
    ),
)

if not RETRIEVAL_READY:
    print("Skipping the routing table: retrieval is not configured.")
else:
    matched = 0
    for question, expected in ROUTING_CASES:
        agent, trace = build_agent()
        print(f"\nQ: {question}")
        routing_result = agent(question)
        chosen = selected_tool_names(routing_result)
        show_result(routing_result)
        if expected in chosen:
            matched += 1
            print(f"   ✅ expected {expected}, used {chosen}")
        else:
            print(f"   ⚠️  expected {expected}, used {chosen or 'no tool'}")

    print(f"\n{matched} of {len(ROUTING_CASES)} questions reached the expected tool.")
    print("A mismatch identifies a tool description to improve.")

## 5. Test a question with a missing fact

**Purpose:** Check how the agent handles information that Neo4j does not contain.

> **Does AnyCompany Cairo Nile View guarantee room availability next weekend?**

The graph stores hotel details and total room capacity. Live room availability is outside this graph. As a result, the agent can identify the hotel but cannot confirm an open room for next weekend.

The tool returns a clear grounding result:

- **`answerable: false`:** Neo4j lacks enough information for an answer.
- **`missing_fact: live_room_availability`:** Current room availability is the missing information.

Run the next cell to inspect this result. The check uses the tool's structured decision because the model's wording can vary.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping the availability question: retrieval is not configured.")
else:
    agent, trace = build_agent()
    print(f"Q: {AVAILABILITY_QUESTION}")
    availability_result = agent(AVAILABILITY_QUESTION)
    show_result(availability_result)

    verdicts = [
        payload["grounding_result"]
        for payload in recorded_payloads(trace)
        if isinstance(payload.get("grounding_result"), dict)
    ]
    print(f"\nTools used: {selected_tool_names(availability_result) or 'none'}")
    print(f"Verdicts returned: {json.dumps(verdicts)}")

    problems = []
    if not trace.calls:
        problems.append("the agent answered an availability question with no tool call")
    if not any(
        verdict.get("missing_fact") == MISSING_LIVE_AVAILABILITY
        for verdict in verdicts
    ):
        problems.append(
            f"no tool result reported missing_fact={MISSING_LIVE_AVAILABILITY}"
        )
    report_problems(problems, "a tool ran and reported live room availability as unsupported.")

## 6. Test a message with no hotel question

**Purpose:** Confirm that a social message receives a direct reply.

> **thanks, that is all**

This message asks for no hotel facts, so the agent can reply without calling a tool. Run the next cell and confirm that the trace contains no tool call.

In [ ]:
SOCIAL_TURN = "thanks, that is all"

if not RETRIEVAL_READY:
    print("Skipping the social turn: retrieval is not configured.")
else:
    agent, trace = build_agent()
    print(f"Q: {SOCIAL_TURN}")
    social_result = agent(SOCIAL_TURN)
    show_result(social_result)

    used = selected_tool_names(social_result)
    if used:
        print(f"   ⚠️  used {used}; a thank-you needs no hotel fact")
    else:
        print("   ✅ no tool call: the model answered without reading the graph.")

## 7. Test the guest-limit rule

**Purpose:** Confirm that Neo4j rejects a booking request for more than 10 guests.

The next cell creates a request for 15 guests. Neo4j checks the rule before saving the request. It returns a rejection and creates no booking record.

This test needs Neo4j only. It uses a known hotel ID, future dates, and a new `request_id`. The next section reuses these values.

In [ ]:
if not NEO4J_READY:
    print("Skipping rule rejection: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    hero_id = manifest.hotels[HERO_SOURCE]
    check_in = (date.today() + timedelta(days=30)).isoformat()
    check_out = (date.today() + timedelta(days=32)).isoformat()
    REQUEST_ID = str(uuid.uuid4())
    print(f"Hero hotel_id from fixture manifest: {hero_id}")
    print(f"Caller-created request_id for retries: {REQUEST_ID}")
    print(f"Stay: {check_in} to {check_out}\n")

    over_limit_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": OVER_LIMIT_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        rejected = create_reservation_request(
            over_limit_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print(json.dumps(rejected, indent=2))

    assert rejected["status"] == ReservationStatus.REJECTED.value, rejected
    assert rejected["reason_code"] == ReservationReason.MAX_GUESTS_EXCEEDED.value, rejected
    assert rejected["hotel_id"] == hero_id, rejected
    assert rejected["max_guests"] == MAX_GUESTS, rejected

## 8. Create one booking request and safely send it again

**Purpose:** Confirm that a repeated request creates only one booking record.

The previous request was rejected before Neo4j saved it. The next cell reuses its hotel ID, dates, and `request_id`. It changes the guest count to 10 and sends the valid request twice.

- **First request:** Create one booking request for the hotel.
- **Repeated request:** Return the saved request with `duplicate=true`.
- **`request_id`:** Identify one booking request with an ID created by the caller. Neo4j saves one request for each ID.

In [ ]:
if not NEO4J_READY:
    print("Skipping valid write: Neo4j is not configured.")
else:
    valid_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": MAX_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        accepted = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
        replay = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print("First delivery:")
    print(json.dumps(accepted, indent=2))
    print("\nSame request_id re-delivered:")
    print(json.dumps(replay, indent=2))

    assert accepted["status"] == ReservationStatus.ACCEPTED.value, accepted
    assert accepted["hotel_id"] == hero_id, accepted
    assert accepted["duplicate"] is False, accepted

    assert replay["status"] == ReservationStatus.ACCEPTED.value, replay
    assert replay["hotel_id"] == hero_id, replay
    assert replay["duplicate"] is True, replay

## 9. Verify the booking request in Neo4j

**Purpose:** Confirm that Neo4j saved one accepted request for one hotel.

Run the next cell to find the request saved in Section 8. The query uses the same `request_id` and returns the request with its connected hotel.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph inspection: Neo4j is not configured.")
else:
    query = (
        "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
        "RETURN r.status AS status, r.guests AS guests, r.check_in AS check_in, "
        "r.check_out AS check_out, h.hotel_id AS hotel_id, h.name AS hotel_name, "
        "toString(r.created_at) AS created_at"
    )
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        with driver.session(database=config.database) as session:
            for record in session.run(query, rid=REQUEST_ID):
                print(dict(record))
    finally:
        driver.close()

## Continue to the next modules

You now have a local agent with two graph tools and a protected booking command.

- **Module 4:** Move the two read tools to AWS Lambda and AgentCore Gateway.
- **Module 5:** Package the agent for AgentCore Runtime.
- **Module 6:** Add memory for each user across sessions, including sources and corrections.